# Adalat-small Frozen Vaani + LAHAJA Evaluation

This notebook performs the **one frozen external evaluation** authorized by
Notebook 14's `pass_to_frozen_benchmarks` gate. It never trains or selects a
checkpoint.

It evaluates the fixed `serious_labelsafe_v1` adapter against:

- the unadapted pinned Adalat checkpoint on frozen Vaani and LAHAJA;
- the pinned ARTPARK checkpoint on frozen Vaani for the predeclared headline gate.

Existing no-mask baseline predictions are reused only if a deterministic,
condition-balanced masked-inference audit produces identical normalized text.
Otherwise that baseline is recomputed through the masked path. Every prediction
is appended directly to Drive, so interrupted runs resume.

**Primary success condition:** adapted Adalat must lower pooled telephone WER
versus base Adalat on both benchmarks with paired-bootstrap 95% intervals
entirely below zero, while original-audio relative WER regression remains at
or below 5%.


## Before Running

1. Select **Runtime > Change runtime type > T4 GPU**.
2. Confirm Notebook 14 completed and this folder exists:
   `MyDrive/call-whisper/results/channel_adaptation_adalat_small_seed0/serious_labelsafe_v1/`
3. Confirm the frozen Vaani and LAHAJA artifact folders from Notebooks 11-13
   remain in `MyDrive/call-whisper/results/`.
4. Run all cells.

Persistent output:

`MyDrive/call-whisper/results/adalat_frozen_evaluation_v1/`

Do not change the adapter, decoding settings, benchmark rows, or gate after
seeing results.


In [ ]:
# Mount Drive, clone the current repository, and install evaluation dependencies.
import importlib
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
if subprocess.run(['nvidia-smi'], check=False).returncode:
    raise RuntimeError('GPU runtime required. Select Runtime > Change runtime type > T4 GPU.')

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/call-whisper')
REPO_DIR = Path('/content/CallWhisper-8k')
os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/anshulLuhsna/CallWhisper-8k.git', str(REPO_DIR),
], check=True)
os.chdir(REPO_DIR)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'
], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.46.3', 'peft==0.13.2', 'accelerate==0.34.2',
    'jiwer>=3.0.4,<5', 'numpy>=1.26,<3', 'pandas>=2,<3',
    'soundfile>=0.12', 'tabulate>=0.9', 'tqdm>=4.66',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'
], check=True)

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
stale_modules = [
    name for name in sys.modules
    if name == 'callwhisper' or name.startswith('callwhisper.')
]
for module_name in stale_modules:
    del sys.modules[module_name]
importlib.invalidate_caches()

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Python:', platform.python_version())
print('Repository commit:', commit)
print('GPU setup complete.')


In [ ]:
# Frozen evaluation contract and artifact preflight.
import hashlib
import json
from importlib.metadata import version

CONDITIONS = (
    'original',
    'bandlimit_8k',
    'bandlimit_8k_g711_alaw',
    'bandlimit_8k_g711_mulaw',
    'bandlimit_8k_gsm_fr',
)
SEED = 0
NUM_BEAMS = 1
LANGUAGE = 'hi'
TASK = 'transcribe'
BOOTSTRAP_REPLICATES = 20_000
AUDIT_ROWS_PER_CONDITION = 4

BASE_MODEL_ID = 'adalat-ai/whisper-small-hi-high-lr'
BASE_MODEL_REVISION = 'e78553113fe7a483dbf82fefb2cbe4ea4b6bf901'
ARTPARK_MODEL_ID = 'ARTPARK-IISc/whisper-medium-vaani-hindi'
ARTPARK_MODEL_REVISION = '8e4d906e0eec66f27a31286e1a034702ef6d11bc'

ADAPTATION_DIR = (
    DRIVE_PROJECT_DIR / 'results/channel_adaptation_adalat_small_seed0/'
    'serious_labelsafe_v1'
)
ADAPTER_DIR = ADAPTATION_DIR / 'final_adapter'
PROCESSOR_DIR = ADAPTATION_DIR / 'processor'
VAANI_INPUT = DRIVE_PROJECT_DIR / 'results/vaani_paired_pilot_v2'
VAANI_PRIOR = DRIVE_PROJECT_DIR / 'results/vaani_paired_model_full_v1'
LAHAJA_CANDIDATES = [
    DRIVE_PROJECT_DIR / 'results/lahaja_paired_external_v2_masked',
    DRIVE_PROJECT_DIR / 'results/lahaja_paired_external_v1',
]
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'results/adalat_frozen_evaluation_v1'
WORK_ROOT = Path('/content/adalat_frozen_evaluation_v1')
for directory in (OUTPUT_DIR, WORK_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

def has_lahaja_contract(root: Path) -> bool:
    manifest_exists = (
        (root / 'frozen_eval_rows.csv').exists()
        or (root / 'lahaja_speaker_132_all_conditions.csv').exists()
    )
    archives_exist = all(
        (root / 'archives' / f'{condition}.tar.gz').exists()
        for condition in CONDITIONS
    )
    base_exists = (
        root / 'adalat_whisper_small_hi_high_lr_predictions.jsonl'
    ).exists()
    return manifest_exists and archives_exist and base_exists

LAHAJA_INPUT = next(
    (root for root in LAHAJA_CANDIDATES if has_lahaja_contract(root)),
    None,
)
if LAHAJA_INPUT is None:
    raise FileNotFoundError(
        'No complete frozen LAHAJA artifact folder found. Expected v2_masked '
        'or v1 with five archives, a frozen manifest, and base predictions.'
    )

required = [
    ADAPTER_DIR / 'adapter_model.safetensors',
    ADAPTER_DIR / 'adapter_config.json',
    PROCESSOR_DIR / 'preprocessor_config.json',
    PROCESSOR_DIR / 'tokenizer_config.json',
    VAANI_INPUT / 'vaani_pilot_500.csv',
    VAANI_INPUT / 'vaani_pilot_500_all_conditions.csv',
    VAANI_PRIOR / 'adalat_whisper_small_hi_high_lr_predictions.jsonl',
    VAANI_PRIOR / 'artpark_medium_vaani_hindi_predictions.jsonl',
]
required.extend(
    VAANI_INPUT / 'archives' / f'{condition}.tar.gz'
    for condition in CONDITIONS
)
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing frozen evaluation prerequisites: {missing}')

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

semantic_config = {
    'evaluation': 'one frozen post-adaptation evaluation; no training',
    'conditions': list(CONDITIONS),
    'base_model_id': BASE_MODEL_ID,
    'base_model_revision': BASE_MODEL_REVISION,
    'artpark_model_id': ARTPARK_MODEL_ID,
    'artpark_model_revision': ARTPARK_MODEL_REVISION,
    'adapter_sha256': sha256_file(ADAPTER_DIR / 'adapter_model.safetensors'),
    'adapter_config_sha256': sha256_file(ADAPTER_DIR / 'adapter_config.json'),
    'vaani_input': str(VAANI_INPUT),
    'vaani_prior': str(VAANI_PRIOR),
    'lahaja_input': str(LAHAJA_INPUT),
    'attention_mask': True,
    'baseline_reuse_policy': (
        'reuse prior predictions only after exact normalized-text agreement '
        'on deterministic condition-balanced masked audit'
    ),
    'audit_rows_per_condition': AUDIT_ROWS_PER_CONDITION,
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'num_beams': NUM_BEAMS,
    'seed': SEED,
    'training_allowed': False,
}
config_path = OUTPUT_DIR / 'run_config.json'
if config_path.exists():
    existing = json.loads(config_path.read_text(encoding='utf-8'))
    if existing.get('semantic_config') != semantic_config:
        durable = list(OUTPUT_DIR.glob('*_predictions.jsonl'))
        if durable:
            raise RuntimeError(
                'Frozen evaluation contract changed after predictions were saved. '
                'Use a new output directory.'
            )
config_path.write_text(
    json.dumps(
        {'semantic_config': semantic_config, 'repo_commit': commit},
        indent=2,
    ) + '\n',
    encoding='utf-8',
)
package_versions = {
    name: version(name)
    for name in ('transformers', 'peft', 'accelerate', 'jiwer', 'numpy', 'pandas')
}
package_versions.update({'python': platform.python_version(), 'repo_commit': commit})
(OUTPUT_DIR / 'package_versions.json').write_text(
    json.dumps(package_versions, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(semantic_config, indent=2))
print('Persistent output:', OUTPUT_DIR)


In [ ]:
# Restart-safe archive, prediction, and validation helpers.
import tarfile

import pandas as pd
from tqdm.auto import tqdm

def safe_extract(archive_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with tarfile.open(archive_path, 'r:gz') as archive:
        members = archive.getmembers()
        for member in members:
            if member.issym() or member.islnk():
                raise RuntimeError(f'Refusing archive link: {member.name}')
            target = (destination / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f'Unsafe archive path: {member.name}')
        for member in tqdm(members, desc=f'Restoring {archive_path.name}'):
            archive.extract(member, destination, filter='data')

def restore_conditions(source_root: Path, destination: Path, expected: int) -> None:
    for condition in CONDITIONS:
        condition_dir = destination / 'paired_audio' / condition
        existing = list(condition_dir.glob('*.wav')) if condition_dir.exists() else []
        if len(existing) != expected:
            if condition_dir.exists():
                shutil.rmtree(condition_dir)
            safe_extract(
                source_root / 'archives' / f'{condition}.tar.gz',
                destination,
            )
        count = len(list(condition_dir.glob('*.wav')))
        if count != expected:
            raise RuntimeError(
                f'Frozen audio count mismatch for {condition}: {count} != {expected}'
            )
        print(condition, 'files=', count)

def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line_number, line in enumerate(
        path.read_text(encoding='utf-8').splitlines(), start=1
    ):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f'Invalid JSONL at {path}:{line_number}') from exc
    return rows

def append_jsonl(path: Path, row: dict) -> None:
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')
        handle.flush()

def expected_pairs(frame: pd.DataFrame) -> set[tuple[str, str]]:
    return {
        (str(row.sample_key), str(row.condition))
        for row in frame.itertuples()
    }

def validate_prediction_rows(
    rows: list[dict],
    expected: set[tuple[str, str]],
    *,
    label: str | None = None,
    allow_partial: bool = False,
) -> set[tuple[str, str]]:
    observed: set[tuple[str, str]] = set()
    for row in rows:
        key = (str(row['sample_key']), str(row['condition']))
        if key in observed:
            raise RuntimeError(f'Duplicate prediction row: {key}')
        if label is not None and str(row['model_label']) != label:
            raise RuntimeError(
                f'Prediction label mismatch: {row["model_label"]} != {label}'
            )
        observed.add(key)
    if not observed.issubset(expected):
        raise RuntimeError(f'Unexpected prediction keys: {sorted(observed - expected)[:5]}')
    if not allow_partial and observed != expected:
        raise RuntimeError(
            f'Incomplete predictions: {len(observed)} != {len(expected)}'
        )
    return observed


In [ ]:
# Restore and validate the frozen 500-speaker Vaani benchmark.
VAANI_WORK = WORK_ROOT / 'vaani'
restore_conditions(VAANI_INPUT, VAANI_WORK, expected=500)

vaani_pilot_df = pd.read_csv(VAANI_INPUT / 'vaani_pilot_500.csv')
vaani_all_df = pd.read_csv(VAANI_INPUT / 'vaani_pilot_500_all_conditions.csv')
vaani_keys = vaani_pilot_df['sample_key'].astype(str).head(500).tolist()
vaani_df = vaani_all_df[
    vaani_all_df['sample_key'].astype(str).isin(vaani_keys)
].copy()
vaani_df['sample_key'] = vaani_df['sample_key'].astype(str)
vaani_df['condition'] = pd.Categorical(
    vaani_df['condition'], categories=CONDITIONS, ordered=True
)
vaani_df['sample_order'] = vaani_df['sample_key'].map(
    {key: index for index, key in enumerate(vaani_keys)}
)
vaani_df = vaani_df.sort_values(
    ['sample_order', 'condition']
).reset_index(drop=True)
vaani_df['resolved_audio_path'] = vaani_df['audio_path'].map(
    lambda path: str(VAANI_WORK / path)
)
assert len(vaani_df) == 500 * len(CONDITIONS)
assert vaani_df.groupby('sample_key')['condition'].nunique().eq(len(CONDITIONS)).all()
assert {'reference_1', 'reference_2', 'reference_3'}.issubset(vaani_df.columns)
assert all(Path(path).exists() for path in vaani_df['resolved_audio_path'])
vaani_df.to_csv(OUTPUT_DIR / 'vaani_frozen_eval_rows.csv', index=False)
print('Vaani speakers:', vaani_df['sample_key'].nunique())
print('Vaani rows per model:', len(vaani_df))


In [ ]:
# Restore and validate the frozen 132-speaker LAHAJA benchmark.
LAHAJA_WORK = WORK_ROOT / 'lahaja'
restore_conditions(LAHAJA_INPUT, LAHAJA_WORK, expected=132)

lahaja_manifest = (
    LAHAJA_INPUT / 'frozen_eval_rows.csv'
    if (LAHAJA_INPUT / 'frozen_eval_rows.csv').exists()
    else LAHAJA_INPUT / 'lahaja_speaker_132_all_conditions.csv'
)
lahaja_df = pd.read_csv(lahaja_manifest)
lahaja_df['sample_key'] = lahaja_df['sample_key'].astype(str)
lahaja_df['condition'] = pd.Categorical(
    lahaja_df['condition'], categories=CONDITIONS, ordered=True
)
lahaja_df = lahaja_df.sort_values(
    ['sample_key', 'condition']
).reset_index(drop=True)
lahaja_df['resolved_audio_path'] = lahaja_df['audio_path'].map(
    lambda path: str(LAHAJA_WORK / path)
)
if 'reference_text' not in lahaja_df.columns:
    raise RuntimeError('LAHAJA frozen manifest lacks reference_text')
assert len(lahaja_df) == 132 * len(CONDITIONS)
assert lahaja_df.groupby('sample_key')['condition'].nunique().eq(len(CONDITIONS)).all()
assert all(Path(path).exists() for path in lahaja_df['resolved_audio_path'])
lahaja_df.to_csv(OUTPUT_DIR / 'lahaja_frozen_eval_rows.csv', index=False)
print('LAHAJA source:', LAHAJA_INPUT)
print('LAHAJA speakers:', lahaja_df['sample_key'].nunique())
print('LAHAJA rows per model:', len(lahaja_df))


In [ ]:
# Shared masked inference and benchmark-specific scoring.
import gc
import time

import soundfile as sf
import torch
from jiwer import cer as jiwer_cer
from peft import PeftModel
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

from callwhisper.eval.multiref import (
    alignment_multireference_score,
    normalize_vaani_text,
    single_reference_score,
)

torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('CUDA GPU required')
DTYPE = torch.float16
print('GPU:', torch.cuda.get_device_name(0))

MODEL_SPECS = {
    'artpark_reference': {
        'label': 'artpark_reference',
        'model_id': ARTPARK_MODEL_ID,
        'revision': ARTPARK_MODEL_REVISION,
        'role': 'artpark',
    },
    'adalat_base': {
        'label': 'adalat_base',
        'model_id': BASE_MODEL_ID,
        'revision': BASE_MODEL_REVISION,
        'role': 'base',
    },
    'adalat_adapted': {
        'label': 'adalat_adapted',
        'model_id': BASE_MODEL_ID,
        'revision': BASE_MODEL_REVISION,
        'role': 'adapted',
    },
}

def load_eval_stack(spec: dict):
    processor_source = (
        PROCESSOR_DIR if spec['role'] in {'base', 'adapted'} else spec['model_id']
    )
    processor_kwargs = (
        {} if spec['role'] in {'base', 'adapted'}
        else {'revision': spec['revision']}
    )
    processor = AutoProcessor.from_pretrained(
        processor_source, **processor_kwargs
    )
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        spec['model_id'],
        revision=spec['revision'],
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    )
    if spec['role'] == 'adapted':
        model = PeftModel.from_pretrained(
            model, ADAPTER_DIR
        ).merge_and_unload()
    model = model.to(DEVICE)
    model.eval()
    model.generation_config.forced_decoder_ids = None
    return model, processor

def transcribe_one(model, processor, audio_path: Path) -> tuple[str, float]:
    audio, sample_rate = sf.read(audio_path, dtype='float32', always_2d=False)
    if sample_rate != 16000 or getattr(audio, 'ndim', 1) != 1:
        raise ValueError(f'Expected mono 16 kHz audio: {audio_path}')
    inputs = processor(
        audio,
        sampling_rate=sample_rate,
        return_tensors='pt',
        return_attention_mask=True,
    )
    input_features = inputs.input_features.to(device=DEVICE, dtype=DTYPE)
    attention_mask = inputs.attention_mask.to(device=DEVICE)
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        predicted_ids = model.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            language=LANGUAGE,
            task=TASK,
            num_beams=NUM_BEAMS,
            do_sample=False,
            max_new_tokens=225,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    hypothesis = processor.batch_decode(
        predicted_ids, skip_special_tokens=True
    )[0].strip()
    return hypothesis, elapsed

def score_prediction(benchmark: str, row: dict, hypothesis: str) -> dict:
    if benchmark == 'vaani':
        references = [
            row['reference_1'], row['reference_2'], row['reference_3']
        ]
        score = alignment_multireference_score(references, hypothesis)
        normalized_reference = normalize_vaani_text(references[0])
        reference_fields = {
            'reference_1': references[0],
            'reference_2': references[1],
            'reference_3': references[2],
        }
    else:
        reference = row['reference_text']
        score = single_reference_score(reference, hypothesis)
        normalized_reference = normalize_vaani_text(reference)
        reference_fields = {'reference_text': reference}
    return {
        **reference_fields,
        'substitutions': score.substitutions,
        'insertions': score.insertions,
        'deletions': score.deletions,
        'errors': score.errors,
        'reference_words': score.reference_words,
        'wer': score.wer,
        'cer': float(jiwer_cer(
            normalized_reference,
            normalize_vaani_text(hypothesis),
        )),
    }

def evaluate_model(
    benchmark: str,
    frame: pd.DataFrame,
    spec: dict,
) -> Path:
    output_path = OUTPUT_DIR / f"{benchmark}_{spec['label']}_predictions.jsonl"
    expected = expected_pairs(frame)
    existing = read_jsonl(output_path)
    complete = validate_prediction_rows(
        existing, expected, label=spec['label'], allow_partial=True
    )
    if complete == expected:
        print(benchmark, spec['label'], 'already complete; skipping model load.')
        return output_path

    model, processor = load_eval_stack(spec)
    pending = [
        row for row in frame.to_dict('records')
        if (str(row['sample_key']), str(row['condition'])) not in complete
    ]
    print(benchmark, spec['label'], 'pending:', len(pending), '/', len(frame))
    for row in tqdm(pending, desc=f"{benchmark}: {spec['label']}"):
        hypothesis, elapsed = transcribe_one(
            model, processor, Path(row['resolved_audio_path'])
        )
        duration = float(row['duration_s'])
        scored = score_prediction(benchmark, row, hypothesis)
        append_jsonl(output_path, {
            'benchmark': benchmark,
            'model_label': spec['label'],
            'model_id': spec['model_id'],
            'model_revision': spec['revision'],
            'adapter_sha256': (
                semantic_config['adapter_sha256']
                if spec['role'] == 'adapted' else None
            ),
            'sample_key': str(row['sample_key']),
            'speaker_id': str(row.get('speaker_id', row.get('sp_id', ''))),
            'condition': str(row['condition']),
            'audio_path': row['audio_path'],
            **scored,
            'hypothesis_text': hypothesis,
            'duration_s': duration,
            'inference_s': elapsed,
            'real_time_factor': elapsed / duration if duration > 0 else None,
            'attention_mask': True,
            'num_beams': NUM_BEAMS,
            'repo_commit': commit,
        })
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    completed = read_jsonl(output_path)
    validate_prediction_rows(completed, expected, label=spec['label'])
    print('Completed:', output_path, 'rows=', len(completed))
    return output_path


In [ ]:
# Audit masked equivalence before reusing old base/ARTPARK predictions.
def prior_prediction_map(path: Path) -> dict[tuple[str, str], dict]:
    rows = read_jsonl(path)
    mapping = {}
    for row in rows:
        key = (str(row['sample_key']), str(row['condition']))
        if key in mapping:
            raise RuntimeError(f'Duplicate prior prediction: {path} {key}')
        mapping[key] = row
    return mapping

def audit_rows(frame: pd.DataFrame) -> list[dict]:
    selected = []
    for condition in CONDITIONS:
        subset = frame[
            frame['condition'].astype(str) == condition
        ].sort_values('sample_key').head(AUDIT_ROWS_PER_CONDITION)
        selected.extend(subset.to_dict('records'))
    return selected

def audit_or_load(
    benchmark: str,
    frame: pd.DataFrame,
    prior_path: Path,
    spec: dict,
) -> dict:
    audit_path = OUTPUT_DIR / f"{benchmark}_{spec['label']}_mask_audit.json"
    prior_hash = sha256_file(prior_path)
    if audit_path.exists():
        audit = json.loads(audit_path.read_text(encoding='utf-8'))
        if audit.get('prior_sha256') != prior_hash:
            raise RuntimeError(f'Prior prediction file changed after audit: {prior_path}')
        print('Reusing completed audit:', audit_path)
        return audit

    expected = expected_pairs(frame)
    prior_map = prior_prediction_map(prior_path)
    if set(prior_map) != expected:
        raise RuntimeError(
            f'Prior prediction coverage mismatch: {prior_path} '
            f'{len(prior_map)} != {len(expected)}'
        )
    model, processor = load_eval_stack(spec)
    comparisons = []
    for row in tqdm(
        audit_rows(frame),
        desc=f"{benchmark}: {spec['label']} mask audit",
    ):
        key = (str(row['sample_key']), str(row['condition']))
        masked_text, _ = transcribe_one(
            model, processor, Path(row['resolved_audio_path'])
        )
        prior_text = str(prior_map[key]['hypothesis_text'])
        comparisons.append({
            'sample_key': key[0],
            'condition': key[1],
            'prior_text': prior_text,
            'masked_text': masked_text,
            'normalized_equal': (
                normalize_vaani_text(prior_text)
                == normalize_vaani_text(masked_text)
            ),
        })
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    changed = sum(not row['normalized_equal'] for row in comparisons)
    audit = {
        'benchmark': benchmark,
        'model_label': spec['label'],
        'prior_path': str(prior_path),
        'prior_sha256': prior_hash,
        'rows_audited': len(comparisons),
        'changed_normalized_predictions': changed,
        'reuse_prior_predictions': changed == 0,
        'comparisons': comparisons,
    }
    audit_path.write_text(
        json.dumps(audit, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    print(json.dumps({
        key: value for key, value in audit.items() if key != 'comparisons'
    }, indent=2))
    return audit

def import_prior_predictions(
    benchmark: str,
    frame: pd.DataFrame,
    prior_path: Path,
    spec: dict,
    audit: dict,
) -> Path:
    output_path = OUTPUT_DIR / f"{benchmark}_{spec['label']}_predictions.jsonl"
    expected = expected_pairs(frame)
    existing = read_jsonl(output_path)
    complete = validate_prediction_rows(
        existing, expected, label=spec['label'], allow_partial=True
    )
    prior_map = prior_prediction_map(prior_path)
    frame_map = {
        (str(row['sample_key']), str(row['condition'])): row
        for row in frame.to_dict('records')
    }
    for key in sorted(expected - complete):
        prior = dict(prior_map[key])
        canonical_score = score_prediction(
            benchmark,
            frame_map[key],
            str(prior['hypothesis_text']),
        )
        prior.update({
            'benchmark': benchmark,
            'model_label': spec['label'],
            'model_id': spec['model_id'],
            'model_revision': spec['revision'],
            **canonical_score,
            'source_prediction_path': str(prior_path),
            'source_prediction_sha256': audit['prior_sha256'],
            'attention_mask': 'reused_after_exact_audit',
            'attention_mask_audit_rows': audit['rows_audited'],
            'repo_commit': commit,
        })
        append_jsonl(output_path, prior)
    rows = read_jsonl(output_path)
    validate_prediction_rows(rows, expected, label=spec['label'])
    print('Imported audited prior predictions:', output_path, 'rows=', len(rows))
    return output_path

def baseline_path(
    benchmark: str,
    frame: pd.DataFrame,
    prior_path: Path,
    spec: dict,
) -> Path:
    output_path = OUTPUT_DIR / f"{benchmark}_{spec['label']}_predictions.jsonl"
    existing = read_jsonl(output_path)
    expected = expected_pairs(frame)
    if existing:
        observed = validate_prediction_rows(
            existing, expected, label=spec['label'], allow_partial=True
        )
        if observed == expected:
            print('Baseline already complete:', output_path)
            return output_path
    audit = audit_or_load(benchmark, frame, prior_path, spec)
    if audit['reuse_prior_predictions']:
        return import_prior_predictions(
            benchmark, frame, prior_path, spec, audit
        )
    print('Masked audit changed predictions; recomputing or resuming full baseline.')
    return evaluate_model(benchmark, frame, spec)

vaani_artpark_path = baseline_path(
    'vaani',
    vaani_df,
    VAANI_PRIOR / 'artpark_medium_vaani_hindi_predictions.jsonl',
    MODEL_SPECS['artpark_reference'],
)
vaani_base_path = baseline_path(
    'vaani',
    vaani_df,
    VAANI_PRIOR / 'adalat_whisper_small_hi_high_lr_predictions.jsonl',
    MODEL_SPECS['adalat_base'],
)
lahaja_base_path = baseline_path(
    'lahaja',
    lahaja_df,
    LAHAJA_INPUT / 'adalat_whisper_small_hi_high_lr_predictions.jsonl',
    MODEL_SPECS['adalat_base'],
)


In [ ]:
# Generate or resume the adapter's frozen Vaani predictions.
vaani_adapted_path = evaluate_model(
    'vaani', vaani_df, MODEL_SPECS['adalat_adapted']
)


In [ ]:
# Generate or resume the adapter's frozen LAHAJA predictions.
lahaja_adapted_path = evaluate_model(
    'lahaja', lahaja_df, MODEL_SPECS['adalat_adapted']
)


In [ ]:
# Build summaries, paired bootstrap intervals, and the predeclared final gate.
from callwhisper.eval.paired_bootstrap import (
    paired_bootstrap_rows,
    write_outputs as write_bootstrap_outputs,
)

def summarize_predictions(paths: list[Path], benchmark: str) -> pd.DataFrame:
    rows = [row for path in paths for row in read_jsonl(path)]
    predictions = pd.DataFrame(rows)
    if predictions.duplicated(['model_label', 'sample_key', 'condition']).any():
        raise RuntimeError(f'Duplicate {benchmark} predictions')
    summary_rows = []
    for label, model_frame in predictions.groupby('model_label', sort=False):
        for condition in CONDITIONS:
            frame = model_frame[model_frame['condition'] == condition]
            summary_rows.append(summarize_frame(frame, label, condition))
        summary_rows.append(summarize_frame(
            model_frame[model_frame['condition'] != 'original'],
            label,
            'pooled_telephone',
        ))
    return pd.DataFrame(summary_rows)

def summarize_frame(frame: pd.DataFrame, model_label: str, slice_name: str) -> dict:
    substitutions = int(frame['substitutions'].sum())
    insertions = int(frame['insertions'].sum())
    deletions = int(frame['deletions'].sum())
    reference_words = int(frame['reference_words'].sum())
    return {
        'model_label': model_label,
        'slice': slice_name,
        'speakers': int(frame['sample_key'].nunique()),
        'files': int(len(frame)),
        'substitutions': substitutions,
        'insertions': insertions,
        'deletions': deletions,
        'reference_words': reference_words,
        'corpus_wer': (
            substitutions + insertions + deletions
        ) / reference_words,
        'macro_utterance_wer': float(frame['wer'].mean()),
        'macro_utterance_cer': float(frame['cer'].mean()),
        'mean_real_time_factor': float(frame['real_time_factor'].mean()),
    }

def bootstrap_pair(
    benchmark: str,
    comparison: str,
    reference_path: Path,
    candidate_path: Path,
) -> list[dict]:
    rows = paired_bootstrap_rows(
        read_jsonl(reference_path),
        read_jsonl(candidate_path),
        replicates=BOOTSTRAP_REPLICATES,
        seed=SEED,
    )
    destination = OUTPUT_DIR / f'{benchmark}_{comparison}_bootstrap'
    write_bootstrap_outputs(rows, destination)
    return rows

def bootstrap_metric(
    rows: list[dict],
    metric: str,
    slice_name: str,
) -> dict:
    matches = [
        row for row in rows
        if row['metric'] == metric and row['slice'] == slice_name
    ]
    if len(matches) != 1:
        raise RuntimeError(f'Missing bootstrap metric: {metric} {slice_name}')
    return matches[0]

def summary_metric(
    frame: pd.DataFrame,
    model_label: str,
    slice_name: str,
) -> float:
    row = frame[
        (frame['model_label'] == model_label)
        & (frame['slice'] == slice_name)
    ]
    if len(row) != 1:
        raise RuntimeError(f'Missing summary metric: {model_label} {slice_name}')
    return float(row.iloc[0]['corpus_wer'])

vaani_summary = summarize_predictions(
    [vaani_artpark_path, vaani_base_path, vaani_adapted_path],
    'vaani',
)
lahaja_summary = summarize_predictions(
    [lahaja_base_path, lahaja_adapted_path],
    'lahaja',
)
vaani_summary.to_csv(OUTPUT_DIR / 'vaani_summary.csv', index=False)
lahaja_summary.to_csv(OUTPUT_DIR / 'lahaja_summary.csv', index=False)
(OUTPUT_DIR / 'vaani_summary.md').write_text(
    vaani_summary.to_markdown(index=False) + '\n', encoding='utf-8'
)
(OUTPUT_DIR / 'lahaja_summary.md').write_text(
    lahaja_summary.to_markdown(index=False) + '\n', encoding='utf-8'
)

vaani_base_bootstrap = bootstrap_pair(
    'vaani', 'base_vs_adapted', vaani_base_path, vaani_adapted_path
)
lahaja_base_bootstrap = bootstrap_pair(
    'lahaja', 'base_vs_adapted', lahaja_base_path, lahaja_adapted_path
)
vaani_artpark_bootstrap = bootstrap_pair(
    'vaani', 'artpark_vs_adapted', vaani_artpark_path, vaani_adapted_path
)

def benchmark_gate(
    benchmark: str,
    summary: pd.DataFrame,
    bootstrap_rows: list[dict],
) -> dict:
    base_original = summary_metric(summary, 'adalat_base', 'original')
    adapted_original = summary_metric(summary, 'adalat_adapted', 'original')
    base_pooled = summary_metric(summary, 'adalat_base', 'pooled_telephone')
    adapted_pooled = summary_metric(summary, 'adalat_adapted', 'pooled_telephone')
    pooled_gap = bootstrap_metric(
        bootstrap_rows, 'model_gap', 'pooled_telephone'
    )
    channel_gap = bootstrap_metric(
        bootstrap_rows, 'channel_penalty_gap', 'pooled_telephone'
    )
    relative_original_regression = (
        (adapted_original - base_original) / base_original
        if base_original > 0 else float('inf')
    )
    if channel_gap['ci_95_upper'] < 0:
        channel_sensitivity = 'statistically_supported_reduction'
    elif channel_gap['ci_95_lower'] > 0:
        channel_sensitivity = 'statistically_supported_increase'
    else:
        channel_sensitivity = 'inconclusive'
    passed = (
        pooled_gap['estimate'] < 0
        and pooled_gap['ci_95_upper'] < 0
        and relative_original_regression <= 0.05
    )
    return {
        'benchmark': benchmark,
        'verdict': 'pass' if passed else 'fail',
        'base_original_wer': base_original,
        'adapted_original_wer': adapted_original,
        'original_relative_wer_regression': relative_original_regression,
        'base_pooled_telephone_wer': base_pooled,
        'adapted_pooled_telephone_wer': adapted_pooled,
        'adapted_minus_base_pooled_wer': pooled_gap['estimate'],
        'adapted_minus_base_pooled_wer_ci_95': [
            pooled_gap['ci_95_lower'], pooled_gap['ci_95_upper']
        ],
        'adapted_minus_base_channel_penalty': channel_gap['estimate'],
        'adapted_minus_base_channel_penalty_ci_95': [
            channel_gap['ci_95_lower'], channel_gap['ci_95_upper']
        ],
        'channel_sensitivity_conclusion': channel_sensitivity,
    }

vaani_gate = benchmark_gate(
    'vaani', vaani_summary, vaani_base_bootstrap
)
lahaja_gate = benchmark_gate(
    'lahaja', lahaja_summary, lahaja_base_bootstrap
)
vaani_artpark_pooled = bootstrap_metric(
    vaani_artpark_bootstrap, 'model_gap', 'pooled_telephone'
)
beats_artpark_vaani = (
    vaani_artpark_pooled['estimate'] < 0
    and vaani_artpark_pooled['ci_95_upper'] < 0
)
external_pass = (
    vaani_gate['verdict'] == 'pass'
    and lahaja_gate['verdict'] == 'pass'
)
final_gate = {
    'verdict': (
        'pass_external_generalization'
        if external_pass else 'fail_external_generalization'
    ),
    'vaani': vaani_gate,
    'lahaja': lahaja_gate,
    'headline_artpark_gate': {
        'verdict': (
            'adapted_beats_artpark_on_frozen_vaani'
            if beats_artpark_vaani else 'not_established'
        ),
        'adapted_minus_artpark_pooled_wer': vaani_artpark_pooled['estimate'],
        'ci_95': [
            vaani_artpark_pooled['ci_95_lower'],
            vaani_artpark_pooled['ci_95_upper'],
        ],
    },
    'rule': (
        'On both Vaani and LAHAJA: adapted pooled telephone WER must improve '
        'versus base with paired 95% CI entirely below zero, and original '
        'relative WER regression must be <=5%.'
    ),
    'claim_boundary': (
        'A pass supports frozen-benchmark generalization for these slices and '
        'transforms; it does not establish robustness for every phone call.'
    ),
}
gate_path = OUTPUT_DIR / 'final_frozen_gate.json'
gate_path.write_text(
    json.dumps(final_gate, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
gate_md = f"""# Adalat Frozen Evaluation Gate

**Overall verdict:** `{final_gate['verdict']}`

- Vaani: `{vaani_gate['verdict']}`
  - pooled adapted-minus-base WER: {vaani_gate['adapted_minus_base_pooled_wer']:.6f}
  - 95% CI: {vaani_gate['adapted_minus_base_pooled_wer_ci_95']}
  - channel sensitivity: `{vaani_gate['channel_sensitivity_conclusion']}`
- LAHAJA: `{lahaja_gate['verdict']}`
  - pooled adapted-minus-base WER: {lahaja_gate['adapted_minus_base_pooled_wer']:.6f}
  - 95% CI: {lahaja_gate['adapted_minus_base_pooled_wer_ci_95']}
  - channel sensitivity: `{lahaja_gate['channel_sensitivity_conclusion']}`
- ARTPARK headline gate: `{final_gate['headline_artpark_gate']['verdict']}`

This is one frozen evaluation. Do not tune the adapter against these results.
"""
(OUTPUT_DIR / 'final_frozen_gate.md').write_text(
    gate_md, encoding='utf-8'
)
print('Vaani summary:')
display(vaani_summary)
print('LAHAJA summary:')
display(lahaja_summary)
print(json.dumps(final_gate, indent=2))


In [ ]:
# Final completeness audit and portable report bundle.
required_files = [
    OUTPUT_DIR / 'run_config.json',
    OUTPUT_DIR / 'package_versions.json',
    OUTPUT_DIR / 'vaani_frozen_eval_rows.csv',
    OUTPUT_DIR / 'lahaja_frozen_eval_rows.csv',
    OUTPUT_DIR / 'vaani_artpark_reference_predictions.jsonl',
    OUTPUT_DIR / 'vaani_adalat_base_predictions.jsonl',
    OUTPUT_DIR / 'vaani_adalat_adapted_predictions.jsonl',
    OUTPUT_DIR / 'lahaja_adalat_base_predictions.jsonl',
    OUTPUT_DIR / 'lahaja_adalat_adapted_predictions.jsonl',
    OUTPUT_DIR / 'vaani_summary.csv',
    OUTPUT_DIR / 'vaani_summary.md',
    OUTPUT_DIR / 'lahaja_summary.csv',
    OUTPUT_DIR / 'lahaja_summary.md',
    OUTPUT_DIR / 'final_frozen_gate.json',
    OUTPUT_DIR / 'final_frozen_gate.md',
]
for directory in (
    OUTPUT_DIR / 'vaani_base_vs_adapted_bootstrap',
    OUTPUT_DIR / 'lahaja_base_vs_adapted_bootstrap',
    OUTPUT_DIR / 'vaani_artpark_vs_adapted_bootstrap',
):
    required_files.extend([
        directory / 'paired_bootstrap_v1.csv',
        directory / 'paired_bootstrap_v1.json',
        directory / 'paired_bootstrap_v1.md',
    ])
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise RuntimeError(f'Frozen evaluation incomplete: {missing}')

artifact_hashes = {
    str(path.relative_to(OUTPUT_DIR)): sha256_file(path)
    for path in required_files
}
hash_path = OUTPUT_DIR / 'artifact_sha256.json'
hash_path.write_text(
    json.dumps(artifact_hashes, indent=2) + '\n', encoding='utf-8'
)
bundle_path = OUTPUT_DIR / 'adalat_frozen_evaluation_v1.tar.gz'
with tarfile.open(bundle_path, 'w:gz') as archive:
    for path in required_files + [hash_path]:
        archive.add(path, arcname=path.relative_to(OUTPUT_DIR))
checksum_path = OUTPUT_DIR / f'{bundle_path.name}.sha256'
checksum_path.write_text(
    f'{sha256_file(bundle_path)}  {bundle_path.name}\n',
    encoding='utf-8',
)
os.sync()

from google.colab import files
files.download(str(bundle_path))

print('COMPLETE')
print('Verdict:', final_gate['verdict'])
print('Gate:', OUTPUT_DIR / 'final_frozen_gate.json')
print('Bundle:', bundle_path)
print('Checksum:', checksum_path)
print('All persistent outputs:', OUTPUT_DIR)


## Reading The Result

Open `final_frozen_gate.json` first.

- `pass_external_generalization`: the adapter improved pooled telephone WER
  versus base Adalat on both frozen benchmarks with paired confidence intervals
  entirely below zero and acceptable original-audio behavior.
- `fail_external_generalization`: the internal GramVaani gain did not satisfy
  the predeclared frozen gate. Do not tune against these benchmarks.
- `adapted_beats_artpark_on_frozen_vaani`: the separate, stricter ARTPARK
  headline gate passed on Vaani.
- `not_established`: do not claim the adapter beats ARTPARK.

Report channel-sensitivity conclusions separately from absolute WER gains.
A model can improve absolute WER without reducing its paired telephone penalty.
